In [1]:
# A_check_and_run.py
import shutil, subprocess, sys
from pathlib import Path

python_exe = Path(r"c:\Users\kaden.mcculloch\AppData\Local\anaconda3\envs\kfmcculloch98\python.exe")
script = Path("run_theis_forward_heads.py")
cwd = Path(".").resolve()

print("CWD:", cwd)
print("python exists:", python_exe.exists())
print("script exists:", script.exists())
print("script absolute:", (cwd / script).resolve())

if not python_exe.exists():
    raise SystemExit("Python executable not found at the path above.")

if not script.exists():
    raise SystemExit("Model script not found in template folder. Copy the correct script into this folder.")

# Try to run the command exactly as PST runs it (no shell)
cmd = [str(python_exe), str(script)]
print("Running (list):", cmd)
proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print("returncode:", proc.returncode)
print("--- STDOUT ---")
print(proc.stdout or "<no stdout>")
print("--- STDERR ---")
print(proc.stderr or "<no stderr>")

# Also try running via shell as a single string (Windows style) to reproduce PEST behavior
cmd_str = f'"{python_exe}" "{script}"'
print("Running (shell):", cmd_str)
proc2 = subprocess.run(cmd_str, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, shell=True)
print("returncode (shell):", proc2.returncode)
print("--- STDOUT (shell) ---")
print(proc2.stdout or "<no stdout>")
print("--- STDERR (shell) ---")
print(proc2.stderr or "<no stderr>")

CWD: C:\Python\Personal\proj6\codes\runs\pest\baseline_template
python exists: True
script exists: True
script absolute: C:\Python\Personal\proj6\codes\runs\pest\baseline_template\run_theis_forward_heads.py
Running (list): ['c:\\Users\\kaden.mcculloch\\AppData\\Local\\anaconda3\\envs\\kfmcculloch98\\python.exe', 'run_theis_forward_heads.py']
returncode: 0
--- STDOUT ---
<no stdout>
--- STDERR ---
<no stderr>
Running (shell): "c:\Users\kaden.mcculloch\AppData\Local\anaconda3\envs\kfmcculloch98\python.exe" "run_theis_forward_heads.py"
returncode (shell): 0
--- STDOUT (shell) ---
<no stdout>
--- STDERR (shell) ---
<no stderr>


In [2]:
# fix_pst_model_command_and_write.py
import pyemu
from pathlib import Path

pst_path = Path("baseline.pst")
if not pst_path.exists():
    raise SystemExit("baseline.pst not found")

pst = pyemu.Pst(str(pst_path))

python_exe = r'c:\Users\kaden.mcculloch\AppData\Local\anaconda3\envs\kfmcculloch98\python.exe'
script = r'run_theis_forward_heads.py'
# include proj6.yml explicitly to be explicit about the input file
cmd = f'"{python_exe}" "{script}" proj6.yml'

# set model_command as a single string in the list (PEST will write it as one line)
pst.model_command = [cmd]
pst.write(str(pst_path))
print("Wrote MODEL_COMMAND:", pst.model_command)

noptmax:2, npar_adj:2, nnz_obs:2
Wrote MODEL_COMMAND: ['"c:\\Users\\kaden.mcculloch\\AppData\\Local\\anaconda3\\envs\\kfmcculloch98\\python.exe" "run_theis_forward_heads.py" proj6.yml']


In [3]:
# run_pest_capture.py
import subprocess
from pathlib import Path

pest_exe = r"C:\Python\Personal\proj6\codes\binaries\PESTPP\windows\pestpp-mou.exe"
pst = "baseline.pst"

print("CWD:", Path.cwd())
print("Invoking:", pest_exe, pst)
proc = subprocess.run([pest_exe, pst], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print("RETURN CODE:", proc.returncode)
print("--- PEST STDOUT ---")
print(proc.stdout or "<no stdout>")
print("--- PEST STDERR ---")
print(proc.stderr or "<no stderr>")

CWD: c:\Python\Personal\proj6\codes\runs\pest\baseline_template
Invoking: C:\Python\Personal\proj6\codes\binaries\PESTPP\windows\pestpp-mou.exe baseline.pst
RETURN CODE: 0
--- PEST STDOUT ---


             pestpp-mou: multi-objective optimization under uncertainty

                   by the PEST++ development team

...processing command line: ' C:\Python\Personal\proj6\codes\binaries\PESTPP\windows\pestpp-mou.exe baseline.pst'
...using serial run manager


version: 5.2.24
binary compiled on Nov 12 2025 at 12:05:03
using control file: "baseline.pst"
in directory: "c:\Python\Personal\proj6\codes\runs\pest\baseline_template"
on host: "TTL-HBJ6674"
on a(n) windows operating system
with release configuration
started at 03/13/26 15:30:04

processing control file baseline.pst
         new model command line: c:\Users\kaden.mcculloch\AppData\Local\anaconda3\envs\kfmcculloch98\python.exe run_theis_forward_heads.py proj6.yml
Note: 3 unused lines in pest control file, see rec file...
checking mo

In [4]:
# make_run_bat.py
from pathlib import Path

bat_text = r'''@echo off
REM wrapper to run the forward model for PEST
REM change directory to folder containing this script (PEST should already run here)
"%~dp0c:\Users\kaden.mcculloch\AppData\Local\anaconda3\envs\kfmcculloch98\python.exe" "%~dp0run_theis_forward_heads.py" proj6.yml
exit /b %errorlevel%
'''
p = Path("run_model.bat")
p.write_text(bat_text)
print("Wrote run_model.bat:", p.resolve())
print("Contents:")
print(p.read_text())

Wrote run_model.bat: C:\Python\Personal\proj6\codes\runs\pest\baseline_template\run_model.bat
Contents:
@echo off
REM wrapper to run the forward model for PEST
REM change directory to folder containing this script (PEST should already run here)
"%~dp0c:\Users\kaden.mcculloch\AppData\Local\anaconda3\envs\kfmcculloch98\python.exe" "%~dp0run_theis_forward_heads.py" proj6.yml
exit /b %errorlevel%



In [5]:
# set_pst_to_bat.py
import pyemu
from pathlib import Path

pst_path = Path("baseline.pst")
if not pst_path.exists():
    raise SystemExit("baseline.pst not found")

pst = pyemu.Pst(str(pst_path))

# Set model_command to call the batch wrapper; no extra quoting needed.
# Put single string in the list as PEST writes one MODEL_COMMAND line.
pst.model_command = [r'run_model.bat']
pst.write(str(pst_path))
print("Wrote PST with MODEL_COMMAND = run_model.bat")

noptmax:2, npar_adj:2, nnz_obs:2
Wrote PST with MODEL_COMMAND = run_model.bat


In [6]:
# run_pest_with_bat.py
import subprocess
from pathlib import Path

pest_exe = r"C:\Python\Personal\proj6\codes\binaries\PESTPP\windows\pestpp-mou.exe"
pst = "baseline.pst"

print("CWD:", Path.cwd())
print("Invoking:", pest_exe, pst)
proc = subprocess.run([pest_exe, pst], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print("RETURN CODE:", proc.returncode)
print("--- PEST STDOUT (tail) ---")
print(proc.stdout[-4000:] if proc.stdout else "<no stdout>")
print("--- PEST STDERR (tail) ---")
print(proc.stderr[-4000:] if proc.stderr else "<no stderr>")

CWD: c:\Python\Personal\proj6\codes\runs\pest\baseline_template
Invoking: C:\Python\Personal\proj6\codes\binaries\PESTPP\windows\pestpp-mou.exe baseline.pst
RETURN CODE: 3221226505
--- PEST STDOUT (tail) ---
tus for command: run_model.bat


-->03/13/26 15:34:27 run complete, took: 0.031 seconds
-->0 of 100 complete, 300 failed





    ---  serial run manager runs summary:  ---    
    0 of 100 complete, 300 failed
    process took : 9.463 seconds





    failed run ids:
0,1,2,3,4,5,6,7,8,9,
    10,11,12,13,14,15,16,17,18,19,
    20,21,22,23,24,25,26,27,28,29,
    30,31,32,33,34,35,36,37,38,39,
    40,41,42,43,44,45,46,47,48,49,
    50,51,52,53,54,55,56,57,58,59,
    60,61,62,63,64,65,66,67,68,69,
    70,71,72,73,74,75,76,77,78,79,
    80,81,82,83,84,85,86,87,88,89,
    90,91,92,93,94,95,96,97,98,99,
    

...failed realizations:  100
...the following par:obs realization runs failed: GEN=0_MEMBER=0:GEN=0_MEMBER=0,GEN=0_MEMBER=1:GEN=0_MEMBER=1,GEN=0_MEMBER=2:GEN=0_MEMBER=2,GEN=0_MEMBER